# Preprocessing Pipeline — Split, Scale, Engineer, Resample

**Task IDs:** T-27  
**Purpose:** Demonstrate the full preprocessing pipeline end-to-end and
verify no data leakage (val/test class ratios unchanged after split).

Findings are recorded in `01-data/Preprocessing Decisions.md` (Obsidian vault).

In [ ]:
import warnings

import pandas as pd
from src.config import load_config
from src.data.load import load_raw
from src.data.preprocess import stratified_split, fit_scaler, apply_scaler, save_splits
from src.features.engineering import engineer
from src.data.imbalance import smote_sampler

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = load_config()
seed = cfg["seed"]
df = load_raw()
print(f"Raw: {len(df):,} rows, {df['Class'].sum()} fraud ({df['Class'].mean()*100:.4f}%)")

## 1. Feature Engineering (ADR-005)

Cyclical time encoding + log_amount, then drop raw Time.

In [ ]:
df_eng = engineer(df)
print(f"Columns after engineering: {len(df_eng.columns)}")
print(f"New columns: hour_sin, hour_cos, log_amount")
print(f"Dropped: Time")
df_eng[["hour_sin", "hour_cos", "log_amount", "Amount", "Class"]].describe()

## 2. Stratified Split (70/15/15)

In [ ]:
train, val, test = stratified_split(
    df_eng,
    val_size=cfg["split"]["val_size"],
    test_size=cfg["split"]["test_size"],
    seed=seed,
)

print(f"Train: {len(train):,} rows, {train['Class'].sum()} fraud ({train['Class'].mean()*100:.4f}%)")
print(f"Val:   {len(val):,} rows, {val['Class'].sum()} fraud ({val['Class'].mean()*100:.4f}%)")
print(f"Test:  {len(test):,} rows, {test['Class'].sum()} fraud ({test['Class'].mean()*100:.4f}%)")
print(f"\nLeakage check — fraud rate should be ~equal across splits:")
print(f"  Raw:  {df['Class'].mean()*100:.4f}%")
print(f"  Train:{train['Class'].mean()*100:.4f}%")
print(f"  Val:  {val['Class'].mean()*100:.4f}%")
print(f"  Test: {test['Class'].mean()*100:.4f}%")

## 3. Scaling (RobustScaler on Amount + log_amount)

Fit on train only, then apply to val and test. V1–V28 are already
PCA-centered and need no scaling.

In [ ]:
scaler = fit_scaler(train)
train = apply_scaler(train, scaler)
val = apply_scaler(val, scaler)
test = apply_scaler(test, scaler)

print("Amount stats after scaling (train):")
print(train[["Amount", "log_amount"]].describe().T[["mean", "std", "min", "max"]])

## 4. Save Processed Splits

In [ ]:
from src.config import get_path

out_dir = get_path(cfg, "processed_dir")
save_splits(train, val, test, out_dir)
print(f"Saved to {out_dir}/")

## 5. Resampling Demo (SMOTE on Train Only)

SMOTE is applied **only to the training fold** — never to val or test.
In production, this goes inside an `imblearn.pipeline.Pipeline` so it's
fold-local during cross-validation.

In [ ]:
X_train = train.drop(columns=["Class"]).values
y_train = train["Class"].values

print(f"Before SMOTE: {len(y_train):,} rows, {y_train.sum()} fraud ({y_train.mean()*100:.2f}%)")

sampler = smote_sampler(sampling_strategy=cfg["imbalance"]["sampling_strategy"], seed=seed)
X_res, y_res = sampler.fit_resample(X_train, y_train)

print(f"After SMOTE:  {len(y_res):,} rows, {y_res.sum()} fraud ({y_res.mean()*100:.2f}%)")
print(f"\nVal/test unchanged (no leakage):")
print(f"  Val:  {val['Class'].mean()*100:.4f}% fraud")
print(f"  Test: {test['Class'].mean()*100:.4f}% fraud")

## 6. Summary

- **Split:** 70/15/15 stratified on Class, seed=42
- **Feature engineering:** cyclical hour (sin/cos) + log_amount, Time dropped (ADR-005)
- **Scaling:** RobustScaler on Amount + log_amount, fit on train only
- **Imbalance:** SMOTE (sampling_strategy=0.1) on train only via imblearn Pipeline
- **Leakage check:** val/test fraud rates match raw data (~0.17%)
- **Output:** `data/processed/{train,val,test}.parquet`